# Week 7 Assignment #5: Aggregating Data
Student: Daniel Foulen

Class: IS 362

Date: 3/13/2026

## Overview

In this assignment, we collect movie ratings from five people on a scale of 1–5.
Not every person has seen every movie, so missing values (NaN) are expected and handled appropriately.

We will:
1. Load the ratings into a pandas DataFrame
2. Display averages per user and per movie
3. Normalize ratings per user (scale to 0–1) and display averages
4. Discuss advantages and disadvantages of normalization
5. (Extra Credit) Standardize ratings per user (z-score) and display averages

## Step 1: Load the Data

We store the ratings directly as a dictionary and load them into a pandas DataFrame.
Each row represents a user; each column represents a movie.
Missing values (movies not seen) are represented as `NaN`.

In [2]:
import pandas as pd
import numpy as np

In [5]:
data = {
    'American Sniper':    [5,   4,   np.nan, np.nan, 5  ],
    'Edge of Tomorrow':   [4,   np.nan, np.nan, np.nan, 4],
    'Groundhog Day':      [3,   3,   4,      2,      4  ],
    'Jurassic World':     [np.nan, 3, 3, np.nan, 2  ],
    'Lost in Translation':[np.nan, np.nan, 4, 4,     3  ],
    'Lucy':               [4,   np.nan, 4, np.nan,   3  ]
}

users = ['Oscar', 'Shasuna', 'Carla', 'Chris', 'Ernest']

In [4]:
df = pd.DataFrame(data, index=users)
df

,American Sniper,Edge of Tomorrow,Groundhog Day,Jurassic World,Lost in Translation,Lucy
Oscar,5.0,4.0,3,NaN,NaN,4.0
Shasuna,4.0,NaN,3,3.0,NaN,NaN
Carla,NaN,NaN,4,NaN,4.0,4.0
Chris,NaN,NaN,2,NaN,4.0,NaN
Ernest,5.0,4.0,4,2.0,3.0,3.0


## Step 2: Average Ratings per User and per Movie

We compute the mean rating for each user (row mean) and each movie (column mean),
skipping NaN values so that unseen movies do not distort the averages.

Add average column per user (row-wise mean, ignoring NaN)

In [6]:
df_with_avg = df.copy()
df_with_avg['User Average'] = df.mean(axis=1)

Compute movie averages (column-wise mean, ignoring NaN) and append as a row


In [7]:
movie_avgs = df.mean(axis=0)
movie_avgs['User Average'] = np.nan  # placeholder for the corner cell

df_display = pd.concat([df_with_avg, movie_avgs.rename('Movie Average').to_frame().T])
df_display.round(2)

,American Sniper,Edge of Tomorrow,Groundhog Day,Jurassic World,Lost in Translation,Lucy,User Average
Oscar,5.00,4.0,3.0,NaN,NaN,4.00,4.00
Shasuna,4.00,NaN,3.0,3.0,NaN,NaN,3.33
Carla,NaN,NaN,4.0,NaN,4.00,4.00,4.00
Chris,NaN,NaN,2.0,NaN,4.00,NaN,3.00
Ernest,5.00,4.0,4.0,2.0,3.00,3.00,3.50
Movie Average,4.67,4.0,3.2,2.5,3.67,3.67,NaN


## Step 3: Normalized Ratings

$$\hat{x} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

This is applied **per user (row-wise)**, so each person's lowest rating becomes 0
and their highest becomes 1. This removes bias from users who tend to rate
everything high or everything low.

NaN values are left as NaN. We only normalize the ratings that exist.

In [9]:
df_normalized = df.apply(
    lambda row: (row - row.min()) / (row.max() - row.min()),
    axis=1
)

df_normalized_display = df_normalized.copy()
df_normalized_display['User Average'] = df_normalized.mean(axis=1)

movie_avgs_norm = df_normalized.mean(axis=0)
movie_avgs_norm['User Average'] = np.nan

df_normalized_display = pd.concat([
    df_normalized_display,
    movie_avgs_norm.rename('Movie Average').to_frame().T
])

df_normalized_display.round(3)

,American Sniper,Edge of Tomorrow,Groundhog Day,Jurassic World,Lost in Translation,Lucy,User Average
Oscar,1.0,0.500,0.000,NaN,NaN,0.500,0.500
Shasuna,1.0,NaN,0.000,0.0,NaN,NaN,0.333
Carla,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Chris,NaN,NaN,0.000,NaN,1.000,NaN,0.500
Ernest,1.0,0.667,0.667,0.0,0.333,0.333,0.500
Movie Average,1.0,0.583,0.167,0.0,0.667,0.417,NaN


## Step 4: Conclusion (Normalized vs. Actual Ratings)

### Advantages of Normalized Ratings

- **Removes individual bias:** Some users are generous raters (consistently give 4s and 5s)
  while others are harsh (mostly 1s and 2s). Normalization puts all users on the same scale,
  making comparisons fairer.
- **Better for collaborative filtering:** Recommender systems benefit from normalized data
  because a 5 from a harsh critic carries different weight than a 5 from a lenient one.
  Normalization accounts for this.
- **Highlights relative preferences:** Even if a user gave all movies a 3 or 4, normalization
  reveals which movies they *relatively* preferred more or less.

### Disadvantages of Normalized Ratings

- **Loss of absolute meaning:** A normalized score of 1.0 doesn't mean the user loved the movie.
  It just means it was their *highest-rated* movie in the set. The original 1–5 scale has
  intuitive meaning; normalized values lose that.
- **Sensitive to small datasets:** If a user only rated two movies, their scores become exactly
  0 and 1 regardless of how close or far apart the original ratings were.
- **Edge case (uniform ratings):** If a user gave every movie the same score, the denominator
  becomes zero and normalization is undefined (division by zero). This must be handled carefully.
- **NaN inflation:** Users with few ratings have their min and max computed on a small sample,
  which may make their normalized scores less meaningful compared to users who rated many movies.


## Step 5 (Extra Credit): Standardized Ratings

**Standardization** (Z-Score Scaling) transforms each user's ratings to have a mean of 0
and a standard deviation of 1, using the formula:

$$z = \frac{x - \mu}{\sigma}$$

Where $\mu$ is the user's mean rating and $\sigma$ is their standard deviation.

This is also applied **per user (row-wise)**.

Unlike normalization, standardization does not bound values to [0, 1]. 

Scores can be negative or greater than 1. What it tells us is how far above or below a user's own average each rating falls.

In [12]:
df_standardized = df.apply(
    lambda row: (row - row.mean()) / row.std(),
    axis=1
)

df_standardized_display = df_standardized.copy()
df_standardized_display['User Average'] = df_standardized.mean(axis=1)

movie_avgs_std = df_standardized.mean(axis=0)
movie_avgs_std['User Average'] = np.nan

df_standardized_display = pd.concat([
    df_standardized_display,
    movie_avgs_std.rename('Movie Average').to_frame().T
])

df_standardized_display.round(3)

,American Sniper,Edge of Tomorrow,Groundhog Day,Jurassic World,Lost in Translation,Lucy,User Average
Oscar,1.225,0.000,-1.225,NaN,NaN,0.000,0.0
Shasuna,1.155,NaN,-0.577,-0.577,NaN,NaN,-0.0
Carla,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Chris,NaN,NaN,-0.707,NaN,0.707,NaN,0.0
Ernest,1.430,0.477,0.477,-1.430,-0.477,-0.477,0.0
Movie Average,1.270,0.238,-0.508,-1.004,0.115,-0.238,NaN


## Summary

| Approach | Scale | Use Case |
|---|---|---|
| Raw Ratings | 1–5 | Intuitive, easy to interpret |
| Normalized | 0–1 | Removes high/low rater bias |
| Standardized | Centered at 0 | Shows deviation from personal average |

Each approach has value depending on the goal. For a recommender system,
normalized or standardized ratings generally produce fairer movie-to-movie
comparisons across users with different rating tendencies.